In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/omr-data-preprocessing


/home/dongmin/userdata/dongmin/omr-data-preprocessing/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import groupby

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21
import musicdiff as md
import converter21 as c21

from omrdp.utils import load_ossq_metadata
from omrdp.utils.bbox import load_bboxs
from omrdp.utils.preview import ImageComparePreview, LMXEPreview

In [3]:
dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets'
dataset_data_dir = dataset_dir / 'data'
dataset_score_dir = dataset_dir / 'scores'

dataset_dir.exists()

True

In [4]:
metadata = load_ossq_metadata(dataset_data_dir / 'scores.yaml')

## Inspect ls-yolo-staff-bbox results

### Scanned

In [7]:
pdf_type = 'scanned'

In [35]:
exceptions = []
# list of dict: { 'bbox_path', 'error_type', 'error' }
# error_type: 'count' or 'class'

for score_id, body in metadata.items():
  sqid = body['sqid']
  sq_dir = dataset_score_dir / body['path']
  
  image_dir = sq_dir / 'images' / pdf_type / 'systemwise'

  bbox_paths = image_dir.glob(f'{sqid}*_yolo_bboxs.txt')
  bbox_paths = sorted(bbox_paths)
  
  for b_p in bbox_paths:
    bboxs = load_bboxs(b_p, with_class=True, without_postprocess=True)
    
    if len(bboxs) != 4:
      exceptions.append({
        'bbox_path': b_p,
        'img_path': b_p.with_name(b_p.name.replace('_yolo_bboxs.txt', '.png')),
        'error_type': 'count',
        'error': len(bboxs),
      })
      continue
    
    classes = [ b[0] for b in bboxs ]
    if classes != [2, 2, 1, 0]:
      exceptions.append({
        'bbox_path': b_p,
        'img_path': b_p.with_name(b_p.name.replace('_yolo_bboxs.txt', '.png')),
        'error_type': 'class',
        'error': classes,
      })
      continue

In [45]:
from omrdp.utils.preview import StaffBboxPreview

data_list = [
  (e['img_path'], e['bbox_path'])
  for e in exceptions
]

StaffBboxPreview(data_list).show()

### Synthetic

In [5]:
pdf_type = 'synthetic'

In [6]:
exceptions = []
# list of dict: { 'bbox_path', 'error_type', 'error' }
# error_type: 'count' or 'class'

for score_id, body in metadata.items():
  sqid = body['sqid']
  sq_dir = dataset_score_dir / body['path']
  
  image_dir = sq_dir / 'images' / pdf_type / 'systemwise'

  bbox_paths = image_dir.glob(f'{sqid}*_yolo_bboxs.txt')
  bbox_paths = sorted(bbox_paths)
  
  for b_p in bbox_paths:
    bboxs = load_bboxs(b_p, with_class=True, without_postprocess=True)
    
    if len(bboxs) != 4:
      exceptions.append({
        'bbox_path': b_p,
        'img_path': b_p.with_name(b_p.name.replace('_yolo_bboxs.txt', '.png')),
        'error_type': 'count',
        'error': len(bboxs),
      })
      continue
    
    classes = [ b[0] for b in bboxs ]
    if classes != [2, 2, 1, 0]:
      exceptions.append({
        'bbox_path': b_p,
        'img_path': b_p.with_name(b_p.name.replace('_yolo_bboxs.txt', '.png')),
        'error_type': 'class',
        'error': classes,
      })
      continue

In [9]:
from omrdp.utils.preview import StaffBboxPreview

data_list = [
  (e['img_path'], e['bbox_path'])
  for e in exceptions
]

StaffBboxPreview(data_list).show()

## Write excluded segments list

In [ ]:
from omrdp.utils.yolo import YOLOInfo

pdf_type = 'scanned'

excluded_segments = set()
for score_id, body in metadata.items():
  sqid = body['sqid']
  sq_dir = dataset_score_dir / body['path']
  
  if pdf_type == 'scanned' and not (sq_dir / f'{sqid}_scanned.csv').exists():
    continue
  
  yolo_infos_path = sq_dir / 'images' / pdf_type / f'{sqid}_yolo_infos.yaml'
  yolo_infos = YOLOInfo.load(yolo_infos_path)
  
  if isinstance(yolo_infos.staff_bbox['ignores'], list):
    excluded_segments.update(yolo_infos.staff_bbox['ignores'])

with open(dataset_dir / f'excluded_segments_{pdf_type}_lmxe_systemwise.txt', 'w') as f:
  for seg in sorted(excluded_segments):
    f.write(f"{seg}\n")